# Cricket Ball Detection and Trajectory Tracking

Detects and tracks the cricket ball in indoor net practice footage using three approaches:
1. YOLOv8 deep learning (sports ball detection)
2. Dense optical flow (motion-based detection)
3. Hough circle detection with colour filtering

**Video source:** GoPro MP4 files (GH010059-GH010065) from indoor cricket nets, stored in Google Drive.

**Camera setup:** Ground-level GoPro inside an indoor practice facility with artificial lighting, netting on sides and ceiling, green artificial turf. Bowler on left, batsman in centre-right.

## 1. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)
print('Python:', sys.version)

In [ ]:
import subprocess

packages = [
    "ultralytics>=8.3.0",
    "opencv-python-headless==4.9.0.80",
    "numpy==1.26.4", # Explicitly install numpy<2.0 as pandas needs numpy.rec
    "pandas",        # Explicitly install pandas to ensure compatibility
    # "matplotlib",
    # "seaborn",
    "supervision==0.19.0",
    "tqdm",
    "pyyaml",
    "lap",
]

for pkg in packages:
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", pkg, "-q"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        print(f"  ok: {pkg}")
    except:
        print(f"  skipped: {pkg}")

# sahi separately
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "sahi==0.11.18", "-q"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    SAHI_AVAILABLE = True
    print("  ok: sahi")
except:
    SAHI_AVAILABLE = False
    print("  sahi not available - will use manual slicing")

print("\nDone")

In [ ]:
import os
import cv2
import time
import json
import math
import shutil
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from collections import deque
from tqdm import tqdm

warnings.filterwarnings("ignore")

if IN_COLAB:
    matplotlib.use("module://matplotlib_inline.backend_inline")

try:
    from ultralytics import YOLO
    YOLO_OK = True
    print("YOLO loaded")
except ImportError:
    YOLO_OK = False
    print("YOLO not available")

try:
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    SAHI_AVAILABLE = True
    print("SAHI loaded")
except ImportError:
    SAHI_AVAILABLE = False
    print("SAHI not available - using manual slicing")

print("OpenCV:", cv2.__version__)

## 2. Mount Google Drive and Set Video Path

Your GoPro clips (GH010059.MP4 to GH010065.MP4) are in Google Drive. Update the folder path below.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted")

In [ ]:


# SET YOUR VIDEO FOLDER HERE
VIDEO_FOLDER = "/content/drive/MyDrive/ball_trajectory/"

video_files = []
if os.path.exists(VIDEO_FOLDER):
    all_f = sorted(os.listdir(VIDEO_FOLDER))
    video_files = [os.path.join(VIDEO_FOLDER, f) for f in all_f if f.upper().endswith(".MP4")]
    print(f"Found {len(video_files)} MP4 files:")
    for v in video_files:
        print(f"  {os.path.basename(v)}  ({os.path.getsize(v)/(1024*1024):.0f} MB)")
else:
    print(f"Folder not found: {VIDEO_FOLDER}")
    print('Run: !find /content/drive/MyDrive -name "GH01*.MP4" -type f')

In [ ]:
# search for files if needed:
# !find /content/drive/MyDrive -name "GH01*.MP4" -type f 2>/dev/null | head -20

## 3. Configuration

Parameters tuned for indoor nets GoPro footage: ground-level camera, warm artificial lighting, red ball against green turf, ball is 10-40px diameter at this distance.

In [ ]:
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VIDEO_INDEX = 0
MAX_FRAMES = 300
FRAME_SKIP = 1

YOLO_MODEL = "yolov8x.pt"
YOLO_CONF = 0.15
YOLO_IMGSZ = 1280
YOLO_BALL_CLASSES = [32]

HOUGH_DP = 1.2
HOUGH_MIN_DIST = 40
HOUGH_PARAM1 = 50
HOUGH_PARAM2 = 15
HOUGH_MIN_R = 4
HOUGH_MAX_R = 35

RED_LOW1 = np.array([0, 60, 60])
RED_HIGH1 = np.array([20, 255, 255])
RED_LOW2 = np.array([160, 60, 60])
RED_HIGH2 = np.array([180, 255, 255])
WHITE_LOW = np.array([0, 0, 160])
WHITE_HIGH = np.array([180, 60, 255])

FLOW_THRESHOLD = 2.5
FLOW_MIN_AREA = 8
FLOW_MAX_AREA = 1200

ROI_ENABLED = True
ROI_TOP = 0.25
ROI_BOTTOM = 0.95
ROI_LEFT = 0.05
ROI_RIGHT = 0.95

W_YOLO = 0.50
W_HOUGH = 0.30
W_FLOW = 0.20
TRAIL_LENGTH = 50

print("Config loaded")

## 4. Load Video Frames

In [ ]:
def load_video(path, max_frames=300, skip=1):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print(f"Cannot open: {path}")
        return None, {}
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    info = {"fps": fps, "total": total, "width": w, "height": h}
    print(f"Video: {w}x{h}, {fps:.1f}fps, {total} frames ({total/max(fps,1):.1f}s)")
    frames = []
    num = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if num % skip == 0:
            frames.append((num, frame))
            if max_frames and len(frames) >= max_frames: break
        num += 1
    cap.release()
    print(f"Loaded {len(frames)} frames")
    return frames, info

video_frames = None
video_info = {}
if video_files:
    idx = min(VIDEO_INDEX, len(video_files)-1)
    print(f"Loading: {os.path.basename(video_files[idx])}")
    video_frames, video_info = load_video(video_files[idx], MAX_FRAMES, FRAME_SKIP)

if video_frames is None:
    print("No video. Generating synthetic indoor frames.")
    np.random.seed(42)
    frames = []
    for i in range(MAX_FRAMES or 120):
        t = i / (MAX_FRAMES or 120)
        h, w = 1080, 1920
        frame = np.full((h, w, 3), (50, 120, 50), np.uint8)
        frame[:h//3, :] = np.array([80, 130, 160], np.uint8)
        noise = np.random.randint(0, 15, frame.shape, np.uint8)
        frame = cv2.add(frame, noise)
        bx = int(200 + t * (w - 500))
        by = int(h * 0.55 - 100 * math.sin(math.pi * t))
        cv2.circle(frame, (bx, by), 10, (40, 40, 190), -1)
        frames.append((i, frame))
    video_frames = frames
    video_info = {"fps": 30, "width": 1920, "height": 1080}
    print(f"Generated {len(frames)} frames")

In [ ]:
if video_frames:
    indices = [0, len(video_frames)//4, len(video_frames)//2, 3*len(video_frames)//4]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, idx in zip(axes, indices):
        if idx < len(video_frames):
            fnum, frame = video_frames[idx]
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"Frame {fnum}")
        ax.axis("off")
    plt.suptitle("Video Sample Frames")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "video_samples.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 5. ROI Definition

For indoor nets the ball travels in the lower-middle area. We exclude the ceiling structure and lights at the top and the extreme side edges.

In [ ]:
def make_indoor_roi(width, height):
    x1 = int(width * ROI_LEFT); x2 = int(width * ROI_RIGHT)
    y1 = int(height * ROI_TOP); y2 = int(height * ROI_BOTTOM)
    return np.array([[x1,y1],[x2,y1],[x2,y2],[x1,y2]], dtype=np.int32)

def is_in_roi(cx, cy, roi_poly):
    if not ROI_ENABLED: return True
    return cv2.pointPolygonTest(roi_poly.astype(np.float32), (float(cx),float(cy)), False) >= 0

def draw_roi(frame, roi_poly):
    ov = frame.copy()
    cv2.fillPoly(ov, [roi_poly], (0,60,0))
    out = cv2.addWeighted(ov, 0.15, frame, 0.85, 0)
    cv2.polylines(out, [roi_poly], True, (0,255,0), 2)
    return out

sample = video_frames[len(video_frames)//2][1]
img_h, img_w = sample.shape[:2]
roi_polygon = make_indoor_roi(img_w, img_h)

preview = draw_roi(sample.copy(), roi_polygon)
plt.figure(figsize=(14, 7))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.title("ROI - green = ball detection zone")
plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roi_preview.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Approach 1: YOLOv8 Ball Detection

In [ ]:
def detect_ball_yolo(frame, model):
    results = model(frame, conf=YOLO_CONF, iou=0.45, imgsz=YOLO_IMGSZ, classes=YOLO_BALL_CLASSES, verbose=False)
    detections = []
    for r in results:
        if r.boxes is None: continue
        for i in range(len(r.boxes)):
            x1,y1,x2,y2 = r.boxes.xyxy[i].cpu().numpy()
            conf = float(r.boxes.conf[i])
            cx, cy = (x1+x2)/2, (y1+y2)/2
            radius = max(x2-x1, y2-y1)/2
            if is_in_roi(cx, cy, roi_polygon):
                detections.append({"x":float(cx),"y":float(cy),"radius":float(max(radius,4)),"confidence":conf,"method":"yolo"})
    # check related classes too
    extra = [29,32,33,47]
    results2 = model(frame, conf=YOLO_CONF, iou=0.45, imgsz=YOLO_IMGSZ, classes=extra, verbose=False)
    for r in results2:
        if r.boxes is None: continue
        for i in range(len(r.boxes)):
            x1,y1,x2,y2 = r.boxes.xyxy[i].cpu().numpy()
            bw, bh = x2-x1, y2-y1
            if bw > 80 or bh > 80: continue
            if bw/max(bh,1) < 0.4 or bw/max(bh,1) > 2.5: continue
            conf = float(r.boxes.conf[i])*0.8
            cx, cy = (x1+x2)/2, (y1+y2)/2
            if not is_in_roi(cx, cy, roi_polygon): continue
            if any(math.sqrt((cx-d["x"])**2+(cy-d["y"])**2)<25 for d in detections): continue
            detections.append({"x":float(cx),"y":float(cy),"radius":float(max(max(bw,bh)/2,4)),"confidence":conf,"method":"yolo"})
    return detections

if YOLO_OK:
    yolo_model = YOLO(YOLO_MODEL)
    print(f"Loaded {YOLO_MODEL}")
    yolo_results = {}
    t0 = time.time()
    for idx, (fnum, frame) in enumerate(video_frames):
        dets = detect_ball_yolo(frame, yolo_model)
        yolo_results[fnum] = dets
        if (idx+1) % 50 == 0: print(f"  [{idx+1}/{len(video_frames)}] {len(dets)} candidates")
    total = sum(len(v) for v in yolo_results.values())
    found = sum(1 for v in yolo_results.values() if v)
    print(f"\nYOLO: {total} candidates, {found}/{len(video_frames)} frames ({time.time()-t0:.1f}s)")
else:
    yolo_results = {fn:[] for fn,_ in video_frames}

## 7. Approach 2: Dense Optical Flow

In [ ]:
def detect_ball_flow(prev_gray, curr_gray):
    flow = cv2.calcOpticalFlowFarneback(prev_gray, curr_gray, None, 0.5, 5, 15, 5, 7, 1.5, 0)
    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
    mask = (mag > FLOW_THRESHOLD).astype(np.uint8)*255
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detections = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < FLOW_MIN_AREA or area > FLOW_MAX_AREA: continue
        perim = cv2.arcLength(cnt, True)
        if perim == 0: continue
        circ = 4*math.pi*area/(perim**2)
        if circ < 0.25: continue
        (cx,cy), radius = cv2.minEnclosingCircle(cnt)
        if not is_in_roi(cx, cy, roi_polygon): continue
        bm = np.zeros_like(mask); cv2.drawContours(bm, [cnt], -1, 255, -1)
        avg_mag = cv2.mean(mag, mask=bm)[0]
        conf = min(1.0, (avg_mag/12.0)*circ)
        detections.append({"x":float(cx),"y":float(cy),"radius":float(max(radius,4)),"confidence":conf,"method":"flow"})
    detections.sort(key=lambda d: d["confidence"], reverse=True)
    vis = np.zeros_like(cv2.cvtColor(curr_gray, cv2.COLOR_GRAY2BGR))
    vis[...,0] = (ang*180/np.pi/2).astype(np.uint8)
    vis[...,1] = 255
    vis[...,2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    vis = cv2.cvtColor(vis, cv2.COLOR_HSV2BGR)
    return detections, vis

flow_results = {}; flow_visuals = {}
t0 = time.time()
for idx in range(1, len(video_frames)):
    fp, fc = video_frames[idx-1], video_frames[idx]
    gp = cv2.cvtColor(fp[1], cv2.COLOR_BGR2GRAY)
    gc = cv2.cvtColor(fc[1], cv2.COLOR_BGR2GRAY)
    dets, vis = detect_ball_flow(gp, gc)
    flow_results[fc[0]] = dets; flow_visuals[fc[0]] = vis
    if idx % 50 == 0: print(f"  [{idx}/{len(video_frames)-1}] {len(dets)} candidates")
if video_frames: flow_results[video_frames[0][0]] = []
total = sum(len(v) for v in flow_results.values())
found = sum(1 for v in flow_results.values() if v)
print(f"\nFlow: {total} candidates, {found}/{len(video_frames)} frames ({time.time()-t0:.1f}s)")

In [ ]:
if flow_visuals:
    mk = list(flow_visuals.keys())[len(flow_visuals)//2]
    mi = len(video_frames)//2
    fig, (a1,a2) = plt.subplots(1,2,figsize=(16,6))
    a1.imshow(cv2.cvtColor(video_frames[mi][1], cv2.COLOR_BGR2RGB)); a1.set_title("Original"); a1.axis("off")
    a2.imshow(cv2.cvtColor(flow_visuals[mk], cv2.COLOR_BGR2RGB)); a2.set_title("Optical Flow"); a2.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "optical_flow_sample.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 8. Approach 3: Hough Circle Detection

In [ ]:
def detect_ball_hough(frame):
    hi, wi = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mr1 = cv2.inRange(hsv, RED_LOW1, RED_HIGH1); mr2 = cv2.inRange(hsv, RED_LOW2, RED_HIGH2)
    mred = cv2.bitwise_or(mr1, mr2); mwh = cv2.inRange(hsv, WHITE_LOW, WHITE_HIGH)
    mball = cv2.bitwise_or(mred, mwh)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mball = cv2.morphologyEx(mball, cv2.MORPH_CLOSE, k); mball = cv2.morphologyEx(mball, cv2.MORPH_OPEN, k)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gb = cv2.GaussianBlur(gray, (9,9), 2)
    gm = cv2.bitwise_and(gb, gb, mask=mball)
    dets = []
    c1 = cv2.HoughCircles(gm, cv2.HOUGH_GRADIENT, HOUGH_DP, HOUGH_MIN_DIST, param1=HOUGH_PARAM1, param2=max(HOUGH_PARAM2-5,8), minRadius=HOUGH_MIN_R, maxRadius=HOUGH_MAX_R)
    if c1 is not None:
        for c in np.uint16(np.around(c1[0])):
            cx,cy,r = int(c[0]),int(c[1]),int(c[2])
            if is_in_roi(cx,cy,roi_polygon):
                dets.append({"x":float(cx),"y":float(cy),"radius":float(r),"confidence":0.7,"method":"hough"})
    c2 = cv2.HoughCircles(gb, cv2.HOUGH_GRADIENT, HOUGH_DP, HOUGH_MIN_DIST, param1=HOUGH_PARAM1, param2=HOUGH_PARAM2, minRadius=HOUGH_MIN_R, maxRadius=HOUGH_MAX_R)
    if c2 is not None:
        for c in np.uint16(np.around(c2[0])):
            cx,cy,r = int(c[0]),int(c[1]),int(c[2])
            if not is_in_roi(cx,cy,roi_polygon): continue
            rm = np.zeros((hi,wi),np.uint8); cv2.circle(rm,(cx,cy),max(r,3),255,-1)
            cs = cv2.mean(mball, mask=rm)[0]/255.0
            conf = 0.3 + 0.4*cs
            if not any(math.sqrt((cx-d["x"])**2+(cy-d["y"])**2)<HOUGH_MIN_DIST for d in dets):
                dets.append({"x":float(cx),"y":float(cy),"radius":float(r),"confidence":conf,"method":"hough"})
    dets.sort(key=lambda d: d["confidence"], reverse=True)
    return dets, mball

hough_results = {}
t0 = time.time()
for idx, (fnum, frame) in enumerate(video_frames):
    dets, _ = detect_ball_hough(frame)
    hough_results[fnum] = dets
    if (idx+1) % 50 == 0: print(f"  [{idx+1}/{len(video_frames)}] {len(dets)} candidates")
total = sum(len(v) for v in hough_results.values())
found = sum(1 for v in hough_results.values() if v)
print(f"\nHough: {total} candidates, {found}/{len(video_frames)} frames ({time.time()-t0:.1f}s)")

In [ ]:
if video_frames:
    mid = len(video_frames)//2; fns, fs = video_frames[mid]
    _, ms = detect_ball_hough(fs)
    fig,(a1,a2,a3) = plt.subplots(1,3,figsize=(18,5))
    a1.imshow(cv2.cvtColor(fs, cv2.COLOR_BGR2RGB)); a1.set_title("Original"); a1.axis("off")
    a2.imshow(ms, cmap="gray"); a2.set_title("Colour Mask"); a2.axis("off")
    an = fs.copy()
    for d in hough_results.get(fns,[]): cv2.circle(an,(int(d["x"]),int(d["y"])),int(d["radius"]),(255,0,0),2)
    a3.imshow(cv2.cvtColor(an, cv2.COLOR_BGR2RGB)); a3.set_title("Hough Detections"); a3.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "hough_sample.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 9. Ensemble Fusion

Combine all three methods using weighted voting. Multi-method agreement boosts confidence.

In [ ]:
def fuse_detections(yd, hd, fd, md=35):
    ad = [(d,W_YOLO) for d in yd]+[(d,W_HOUGH) for d in hd]+[(d,W_FLOW) for d in fd]
    if not ad: return None
    used = set(); groups = []
    for i,(di,wi) in enumerate(ad):
        if i in used: continue
        g = [(di,wi)]; used.add(i)
        for j,(dj,wj) in enumerate(ad):
            if j in used: continue
            if math.sqrt((di["x"]-dj["x"])**2+(di["y"]-dj["y"])**2) < md:
                g.append((dj,wj)); used.add(j)
        groups.append(g)
    best = None; bs = -1
    for g in groups:
        tw = sum(w for _,w in g)
        ax = sum(d["x"]*w for d,w in g)/tw
        ay = sum(d["y"]*w for d,w in g)/tw
        ar = sum(d["radius"]*w for d,w in g)/tw
        ms = set(d["method"] for d,_ in g)
        bc = sum(d["confidence"]*w for d,w in g)/tw
        sc = min(1.0, bc + 0.15*(len(ms)-1))
        if sc > bs:
            bs = sc
            best = {"x":ax,"y":ay,"radius":ar,"confidence":sc,"method":f"ensemble({'+'.join(sorted(ms))})"}
    return best

ensemble_results = {}; trajectory = deque(maxlen=TRAIL_LENGTH)
for fnum, frame in video_frames:
    best = fuse_detections(yolo_results.get(fnum,[]), hough_results.get(fnum,[]), flow_results.get(fnum,[]))
    ensemble_results[fnum] = best
    trajectory.append((int(best["x"]),int(best["y"])) if best else None)

found = sum(1 for v in ensemble_results.values() if v)
print(f"Ensemble: ball in {found}/{len(video_frames)} frames")
mc = {}
for v in ensemble_results.values():
    if v: mc[v["method"]] = mc.get(v["method"],0)+1
for m,c in sorted(mc.items(), key=lambda x:-x[1]): print(f"  {m}: {c}")

## 10. Output Video with Bounding Box and Trajectory

In [ ]:
# fix numpy compatibility issue
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==1.26.4", "pandas>=2.0", "-q", "--force-reinstall"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Fixed. Now restart runtime: Runtime > Restart session, then run all cells again.")

In [ ]:
def draw_det(frame, det):
    colors = {"yolo":(0,255,0),"hough":(255,0,0),"flow":(0,165,255)}
    c = (0,255,255)
    for k in colors:
        if k in det["method"]: c = colors[k]; break
    r = max(int(det["radius"]),6); x,y = int(det["x"]),int(det["y"])
    cv2.rectangle(frame,(x-r,y-r),(x+r,y+r),c,2); cv2.circle(frame,(x,y),4,c,-1)
    cv2.putText(frame,f"{det['method'][:20]} {det['confidence']:.2f}",(x-r,y-r-8),cv2.FONT_HERSHEY_SIMPLEX,0.4,c,1)

def draw_trail(frame, pts, color=(0,255,255), ml=50):
    p = list(pts)[-ml:]
    for i in range(1, len(p)):
        if p[i] is None or p[i-1] is None: continue
        a = i/len(p); th = max(1,int(3*a))
        cv2.line(frame, p[i-1], p[i], tuple(int(v*a) for v in color), th)

if video_frames:
    ho,wo = video_frames[0][1].shape[:2]
    outp = os.path.join(OUTPUT_DIR, "ball_detection_output.mp4")
    writer = cv2.VideoWriter(outp, cv2.VideoWriter_fourcc(*"mp4v"), video_info.get("fps",30), (wo,ho))
    trail = deque(maxlen=TRAIL_LENGTH); csv_rows = []
    for idx,(fnum,frame) in enumerate(video_frames):
        ann = frame.copy(); det = ensemble_results.get(fnum)
        if det:
            trail.append((int(det["x"]),int(det["y"]))); draw_det(ann, det)
            csv_rows.append({"frame":fnum,"x":round(det["x"],1),"y":round(det["y"],1),"radius":round(det["radius"],1),"confidence":round(det["confidence"],3),"method":det["method"]})
        else:
            trail.append(None)
            csv_rows.append({"frame":fnum,"x":None,"y":None,"radius":None,"confidence":0,"method":"none"})
        draw_trail(ann, trail)
        for d in yolo_results.get(fnum,[]): cv2.circle(ann,(int(d["x"]),int(d["y"])),3,(0,255,0),1)
        for d in hough_results.get(fnum,[]): cv2.circle(ann,(int(d["x"]),int(d["y"])),3,(255,0,0),1)
        for d in flow_results.get(fnum,[]): cv2.circle(ann,(int(d["x"]),int(d["y"])),3,(0,165,255),1)
        cv2.rectangle(ann,(10,10),(380,120),(0,0,0),-1)
        info = [f"Frame: {fnum}",f"Ball: {'FOUND' if det else 'NOT FOUND'}",f"Method: {det['method'][:30] if det else 'N/A'}",f"Conf: {det['confidence']:.2f}" if det else "Conf: N/A"]
        for i,l in enumerate(info):
            cv2.putText(ann,l,(20,32+i*25),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0) if det else (0,0,255),1)
        writer.write(ann)
    writer.release()
    pd.DataFrame(csv_rows).to_csv(os.path.join(OUTPUT_DIR,"ball_detections.csv"),index=False)
    print(f"Video: {outp}")
    print(f"CSV: output/ball_detections.csv ({len(csv_rows)} rows)")

In [ ]:
if video_frames:
    indices = np.linspace(0, len(video_frames)-1, min(6, len(video_frames)), dtype=int)
    fig, axes = plt.subplots(2,3,figsize=(18,10)); axes = axes.flatten()
    for ai, vi in enumerate(indices):
        fnum, frame = video_frames[vi]; ann = frame.copy()
        det = ensemble_results.get(fnum)
        t = deque(maxlen=TRAIL_LENGTH)
        for vj in range(vi+1):
            fn = video_frames[vj][0]; d = ensemble_results.get(fn)
            t.append((int(d["x"]),int(d["y"])) if d else None)
        if det: draw_det(ann, det)
        draw_trail(ann, t)
        axes[ai].imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        axes[ai].set_title(f"Frame {fnum} - {'found' if det else 'no ball'}", fontsize=10)
        axes[ai].axis("off")
    plt.suptitle("Ball Detection with Bounding Box and Trajectory", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "detection_samples.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 11. Method Comparison

In [ ]:
methods = {"YOLO":yolo_results,"Optical Flow":flow_results,"Hough Circle":hough_results}
rows = []
for name, res in methods.items():
    tc = sum(len(v) for v in res.values()); ff = sum(1 for v in res.values() if v)
    rows.append({"method":name,"candidates":tc,"found":ff,"rate":round(ff/max(len(video_frames),1)*100,1)})
ef = sum(1 for v in ensemble_results.values() if v)
rows.append({"method":"Ensemble","candidates":ef,"found":ef,"rate":round(ef/max(len(video_frames),1)*100,1)})
dfc = pd.DataFrame(rows)
print(dfc.to_string(index=False))

fig, ax = plt.subplots(figsize=(8,4))
colors = ["#2ecc71","#e67e22","#3498db","#e74c3c"]
bars = ax.barh(dfc["method"], dfc["rate"], color=colors[:len(dfc)])
ax.set_xlabel("Detection Rate (%)"); ax.set_title("Ball Detection Rate by Method"); ax.set_xlim(0,105)
for b,v in zip(bars,dfc["rate"]): ax.text(b.get_width()+1,b.get_y()+b.get_height()/2,f"{v:.1f}%",va="center",fontweight="bold")
ax.grid(axis="x",alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"method_comparison.png"),dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
positions = [(d["x"],d["y"]) for d in ensemble_results.values() if d]
if positions:
    xs, ys = zip(*positions)
    fig, ax = plt.subplots(figsize=(12,6))
    sc = ax.scatter(xs, ys, c=range(len(xs)), cmap="coolwarm", s=30, alpha=0.7, edgecolors="white", linewidths=0.5)
    ax.plot(xs, ys, "gray", alpha=0.3, linewidth=1)
    ax.scatter([xs[0]],[ys[0]],c="green",s=100,marker="^",zorder=5,label="Start")
    ax.scatter([xs[-1]],[ys[-1]],c="red",s=100,marker="v",zorder=5,label="End")
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_title("Ball Trajectory")
    ax.invert_yaxis(); ax.legend(); ax.grid(alpha=0.3)
    plt.colorbar(sc,label="Frame"); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,"ball_trajectory.png"),dpi=150,bbox_inches="tight"); plt.show()

## 12. Download Results

In [ ]:
if IN_COLAB:
    from google.colab import files
    import zipfile
    zp = "ball_detection_results.zip"
    with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as zf:
        for root,dirs,fnames in os.walk(OUTPUT_DIR):
            for fn in fnames: fp = os.path.join(root,fn); zf.write(fp,os.path.relpath(fp,"."))
    print(f"Zipped: {zp}"); files.download(zp)
else:
    print("Results in output/")

## 13. Summary

| Output | Description |
|---|---|
| output/ball_detection_output.mp4 | Video with bounding box + trajectory |
| output/ball_detections.csv | Frame-by-frame ball positions |
| output/optical_flow_sample.png | Flow visualisation |
| output/hough_sample.png | Hough circle detection |
| output/detection_samples.png | Annotated frame samples |
| output/method_comparison.png | Method comparison chart |
| output/ball_trajectory.png | Ball trajectory plot |

| Supervisor Requirement | Section |
|---|---|
| Deep learning (YOLO) | Section 6 |
| Optical flow | Section 7 |
| Shape detection (circles) | Section 8 |
| Combine ML + traditional | Section 9 |
| Accuracy over speed | All (low threshold, high res) |
| Start with short clip | Section 4 (MAX_FRAMES=300) |
| Bounding box on video | Section 10 |
| Ball trajectory | Section 10 |